# 数据读取 （data from Unit03_1_5_select.mat）

## 原始数据的读取

In [25]:
import scipy.io

# 读取 .mat 文件
mat_data = scipy.io.loadmat('/home/charles/HZU/Data_processed/multi-condition-transfer-learning/Unit03_1_5_select_1.mat')

# 输出所有键（变量名），查看文件包含的内容
print("Keys in the .mat file:", mat_data.keys())

# 访问实际数据
data = mat_data['Unit03_1_5_select_1']

# 查看数据
print(data)

# 检查数据的类型和形状
print(type(data))
print(data.shape if hasattr(data, 'shape') else 'No shape attribute')

# # 如果 data 是数组或矩阵，查看其部分内容
# print(data[:10])  # 查看前10个元素（如果是数组）


Keys in the .mat file: dict_keys(['__header__', '__version__', '__globals__', 'Unit03_1_5_select_1'])
[[8.98073101e-17 5.93493752e-17 3.33313685e-04 ... 2.81425457e+01
  2.87944984e+01 2.02089348e+01]
 [8.98073101e-17 5.93493752e-17 3.33619362e-04 ... 2.81644821e+01
  2.87728329e+01 2.02019787e+01]
 [8.98073101e-17 5.93493752e-17 3.22607171e-04 ... 2.80990543e+01
  2.87741718e+01 2.02341537e+01]
 ...
 [6.47320175e+00 1.03088837e+01 2.04899997e-01 ... 7.49248535e+02
  7.49325623e+02 6.19291782e+00]
 [6.47659063e+00 1.03162317e+01 2.04046920e-01 ... 7.49166687e+02
  7.49268921e+02 8.11617279e+00]
 [6.46501923e+00 1.02695360e+01 2.03084230e-01 ... 7.49103699e+02
  7.49114685e+02 7.95719814e+00]]
<class 'numpy.ndarray'>
(10000, 14)


## 特征和标签的分离

In [26]:
import numpy as np

# 假设 'data' 是一个二维数组或矩阵
# 分离特征和标签

# 特征是除了最后一列的数据
X = data[:, :-1]  # 所有行，去除最后一列

# 标签是最后一列的数据
y = data[:, -1]  # 所有行，只取最后一列
y = y.reshape(-1, 1)

# # 查看特征和标签
# print("Features (X):")
# print(X[:5])  # 查看前5个特征样本
# print("Labels (y):")
# print(y[:5])  # 查看前5个标签

# 查看特征和标签的形状
print("Shape of Features (X):", X.shape)
print("Shape of Labels (y):", y.shape)

Shape of Features (X): (10000, 13)
Shape of Labels (y): (10000, 1)


## 三集划分

In [27]:
import numpy as np
from sklearn.model_selection import train_test_split

# 假设 X 和 y 是已经分离好的特征和标签
# X: 特征数据，y: 标签数据

# 设置随机种子，确保结果可复现
random_seed = 42

# 控制三集的划分比例：例如 70% 训练集，15% 验证集，15% 测试集
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

# 确保划分比例之和为1
assert train_ratio + val_ratio + test_ratio == 1.0, "The sum of ratios must be 1."

# 第一次划分，将训练集和验证+测试集合并
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=1 - train_ratio, random_state=random_seed)

# 第二次划分，将验证集和测试集分开
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=test_ratio / (val_ratio + test_ratio), random_state=random_seed)

# 打印各个数据集的形状
print("Shape of Training Set (X_train, y_train):", X_train.shape, y_train.shape)
print("Shape of Validation Set (X_val, y_val):", X_val.shape, y_val.shape)
print("Shape of Test Set (X_test, y_test):", X_test.shape, y_test.shape)


Shape of Training Set (X_train, y_train): (6999, 13) (6999, 1)
Shape of Validation Set (X_val, y_val): (1500, 13) (1500, 1)
Shape of Test Set (X_test, y_test): (1501, 13) (1501, 1)


# 图结构搭建

In [28]:
import numpy as np
import torch
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity

# =========================================================
# 输入：
#   X : numpy.ndarray or torch.Tensor
#       shape = [num_samples, num_features]
# 输出：
#   adj : torch.Tensor
#       shape = [num_features, num_features]
# =========================================================

def build_feature_graph_from_X(X, threshold=0.8, device="cpu"):
    """
    Automatically build a feature (column-wise) graph from X.

    Parameters
    ----------
    X : np.ndarray or torch.Tensor
        Shape [num_samples, num_features]
    threshold : float
        Cosine similarity threshold for edge creation
    device : str or torch.device
        cpu / cuda

    Returns
    -------
    adj : torch.Tensor
        Adjacency matrix of shape [num_features, num_features]
    """

    # ---------- 1️⃣ 统一成 numpy ----------
    if isinstance(X, torch.Tensor):
        X_np = X.detach().cpu().numpy()
    else:
        X_np = X

    # ---------- 2️⃣ 自动读取形状 ----------
    num_samples, num_features = X_np.shape
    print(f"[INFO] X shape: samples={num_samples}, features={num_features}")

    # ---------- 3️⃣ 特征（列）作为节点 ----------
    # 每一列是一个节点向量（跨样本）
    X_feature = X_np.T                         # [F, M]

    # ---------- 4️⃣ 特征间相似度 ----------
    sim_matrix = cosine_similarity(X_feature) # [F, F]

    # ---------- 5️⃣ 构建 NetworkX 图 ----------
    G = nx.Graph()
    G.add_nodes_from(range(num_features))

    for i in range(num_features):
        for j in range(i + 1, num_features):
            if sim_matrix[i, j] >= threshold:
                G.add_edge(i, j, weight=sim_matrix[i, j])

    print(f"[INFO] Graph built: nodes={G.number_of_nodes()}, edges={G.number_of_edges()}")

    # ---------- 6️⃣ Graph → 邻接矩阵 ----------
    adj_np = nx.to_numpy_array(G, weight="weight")  # [F, F]

    # ---------- 7️⃣ 转成 torch.Tensor ----------
    adj = torch.tensor(adj_np, dtype=torch.float32, device=device)

    # ---------- 8️⃣ 简单健壮性检查 ----------
    isolated = (adj.sum(dim=1) == 0).sum().item()
    if isolated > 0:
        print(f"[WARN] {isolated} isolated feature nodes detected "
              f"(consider lowering threshold or using KNN graph)")

    print(f"[INFO] adj shape: {adj.shape}")
    return adj


device = "cuda" if torch.cuda.is_available() else "cpu"

adj = build_feature_graph_from_X(X, threshold=0.8, device=device)

print(adj)


[INFO] X shape: samples=10000, features=13
[INFO] Graph built: nodes=13, edges=64
[WARN] 1 isolated feature nodes detected (consider lowering threshold or using KNN graph)
[INFO] adj shape: torch.Size([13, 13])
tensor([[0.0000, 0.8760, 0.8800, 0.0000, 0.9239, 0.9476, 0.9705, 0.9846, 0.9865,
         0.9854, 0.9801, 0.9807, 0.9814],
        [0.8760, 0.0000, 0.9982, 0.0000, 0.9154, 0.0000, 0.8181, 0.8555, 0.8607,
         0.8580, 0.8287, 0.8292, 0.8310],
        [0.8800, 0.9982, 0.0000, 0.0000, 0.9189, 0.0000, 0.8224, 0.8596, 0.8648,
         0.8621, 0.8334, 0.8339, 0.8357],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000],
        [0.9239, 0.9154, 0.9189, 0.0000, 0.0000, 0.9026, 0.9138, 0.9308, 0.9317,
         0.9330, 0.9169, 0.9155, 0.9174],
        [0.9476, 0.0000, 0.0000, 0.0000, 0.9026, 0.0000, 0.9952, 0.9857, 0.9830,
         0.9849, 0.9870, 0.9861, 0.9855],
        [0.9705, 0.8181, 0.8224, 0.0000, 0.9138, 0.

# 模型

In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# ======================================================
# 0️⃣ 固定随机种子
# ======================================================
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ======================================================
# 1️⃣ 工具函数：统一转成 torch.Tensor
# ======================================================
def to_tensor(x, device):
    if isinstance(x, np.ndarray):
        return torch.tensor(x, dtype=torch.float32, device=device)
    elif isinstance(x, torch.Tensor):
        return x.to(device)
    else:
        raise TypeError(f"Unsupported type: {type(x)}")

# ======================================================
# 2️⃣ GCN Layer（adj = F × F）
# ======================================================
class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, x, adj):
        """
        x   : [F, in_dim]
        adj : [F, F]
        """
        F_dim = adj.shape[0]

        I = torch.eye(F_dim, device=adj.device)
        A_hat = adj + I

        deg = A_hat.sum(dim=1)
        deg_inv_sqrt = torch.pow(deg, -0.5)
        deg_inv_sqrt[torch.isinf(deg_inv_sqrt)] = 0.0
        D_inv_sqrt = torch.diag(deg_inv_sqrt)

        A_norm = D_inv_sqrt @ A_hat @ D_inv_sqrt
        out = A_norm @ x
        out = self.linear(out)
        return out

# ======================================================
# 3️⃣ 样本级 GCN 回归模型
# ======================================================
class SampleLevelGCNRegressor(nn.Module):
    def __init__(self, hidden_dim=64, out_dim=1):
        super().__init__()

        self.gcn1 = GCNLayer(1, hidden_dim)
        self.gcn2 = GCNLayer(hidden_dim, hidden_dim)

        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, X_batch, adj):
        """
        X_batch : [B, F]
        adj     : [F, F]
        """
        B, F_dim = X_batch.shape
        outputs = []

        for b in range(B):
            x = X_batch[b].unsqueeze(1)      # [F,1]

            h = F.relu(self.gcn1(x, adj))    # [F,d]
            h = F.relu(self.gcn2(h, adj))    # [F,d]

            g = h.mean(dim=0)                # [d]
            y_hat = self.regressor(g)        # [out_dim]
            outputs.append(y_hat)

        return torch.stack(outputs, dim=0)   # [B,out_dim]

# ======================================================
# 4️⃣ 训练 / 验证函数
# ======================================================
def train_epoch(model, optimizer, criterion, X_data, y_data, adj, batch_size):
    model.train()
    num_samples = X_data.shape[0]
    perm = torch.randperm(num_samples, device=X_data.device)

    total_loss = 0.0

    for start in range(0, num_samples, batch_size):
        idx = perm[start:start + batch_size]
        Xb = X_data[idx]
        yb = y_data[idx]

        y_hat = model(Xb, adj)
        loss = criterion(y_hat, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * Xb.shape[0]

    return total_loss / num_samples


@torch.no_grad()
def eval_epoch(model, criterion, X_data, y_data, adj, batch_size):
    model.eval()
    num_samples = X_data.shape[0]
    total_loss = 0.0

    for start in range(0, num_samples, batch_size):
        Xb = X_data[start:start + batch_size]
        yb = y_data[start:start + batch_size]

        y_hat = model(Xb, adj)
        loss = criterion(y_hat, yb)

        total_loss += loss.item() * Xb.shape[0]

    return total_loss / num_samples

# ======================================================
# 6️⃣ 数据转 Tensor（关键！）
# ======================================================
X_train = to_tensor(X_train, device)
X_val   = to_tensor(X_val, device)
X_test  = to_tensor(X_test, device)

y_train = to_tensor(y_train, device)
y_val   = to_tensor(y_val, device)
y_test  = to_tensor(y_test, device)

adj = to_tensor(adj, device)

# 防呆检查
assert X_train.shape[1] == adj.shape[0], \
    f"Feature mismatch: X has {X_train.shape[1]}, adj is {adj.shape}"

# ======================================================
# 7️⃣ 训练配置
# ======================================================
model = SampleLevelGCNRegressor(hidden_dim=64, out_dim=1).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.MSELoss()

batch_size = 32
epochs = 50

# ======================================================
# 8️⃣ 正式训练
# ======================================================
best_val = float("inf")
best_state = None

for epoch in range(1, epochs + 1):
    train_loss = train_epoch(
        model, optimizer, criterion,
        X_train, y_train, adj, batch_size
    )
    val_loss = eval_epoch(
        model, criterion,
        X_val, y_val, adj, batch_size
    )

    if val_loss < best_val:
        best_val = val_loss
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}

    if epoch == 1 or epoch % 5 == 0:
        print(f"[Epoch {epoch:03d}] "
              f"Train MSE: {train_loss:.4f} | Val MSE: {val_loss:.4f}")

# ======================================================
# 9️⃣ 测试集评估
# ======================================================
model.load_state_dict(best_state)
model.to(device)

test_loss = eval_epoch(
    model, criterion,
    X_test, y_test, adj, batch_size
)

print(f"\n✅ Test MSE: {test_loss:.4f}")


[Epoch 001] Train MSE: 136.0549 | Val MSE: 126.3293
[Epoch 005] Train MSE: 39.2784 | Val MSE: 28.2419
[Epoch 010] Train MSE: 7.4677 | Val MSE: 6.3855
[Epoch 015] Train MSE: 4.4617 | Val MSE: 4.1133
[Epoch 020] Train MSE: 3.4485 | Val MSE: 2.9188
[Epoch 025] Train MSE: 3.0487 | Val MSE: 3.4388
[Epoch 030] Train MSE: 2.5260 | Val MSE: 2.4883
[Epoch 035] Train MSE: 2.5011 | Val MSE: 2.6056
[Epoch 040] Train MSE: 2.4957 | Val MSE: 2.4895
[Epoch 045] Train MSE: 2.4143 | Val MSE: 2.4261
[Epoch 050] Train MSE: 2.4964 | Val MSE: 2.2876

✅ Test MSE: 1.9402


In [32]:
save_path = "/home/charles/HZU/Industrial_Software_Testing/Industrial_Software_Testing/multi_condition_transfer_learning/Single_condition_Regression/result/model_save/best_gcn_model.pth"
torch.save(best_state, save_path)
print(f"Model saved to {save_path}")


Model saved to /home/charles/HZU/Industrial_Software_Testing/Industrial_Software_Testing/multi_condition_transfer_learning/Single_condition_Regression/result/model_save/best_gcn_model.pth


# 测试

In [33]:
from sklearn.metrics import r2_score
import numpy as np
import torch

@torch.no_grad()
def evaluate_r2(model, X_data, y_data, adj, batch_size):
    model.eval()

    y_true_list = []
    y_pred_list = []

    num_samples = X_data.shape[0]

    for start in range(0, num_samples, batch_size):
        Xb = X_data[start:start + batch_size]
        yb = y_data[start:start + batch_size]

        y_hat = model(Xb, adj)  # [B, 1]

        # 🔑 关键：全部拉平成一维向量
        y_true_list.append(yb.view(-1).cpu().numpy())
        y_pred_list.append(y_hat.view(-1).cpu().numpy())

    # 一维拼接（不会受 batch 大小影响）
    y_true = np.concatenate(y_true_list, axis=0)
    y_pred = np.concatenate(y_pred_list, axis=0)

    r2 = r2_score(y_true, y_pred)
    return r2


r2 = evaluate_r2(
    model,
    X_test,
    y_test,
    adj,
    batch_size=batch_size
)

print(f"✅ Test R²: {r2:.4f}")


✅ Test R²: 0.9358
